# Module 1 Lab — Spark & Delta Lake Fundamentals

In this lab you will run real Spark, watch how it executes your code, and create a Delta table you can time-travel and optimize.

## Before you start
1. Create a Lakehouse called **`training_lh`** (or reuse one).
2. Attach it to this notebook (**Add Lakehouse** in the Explorer) and set it as the **default**.
3. Open the **Spark UI** and **Monitoring** from the notebook toolbar — you will use them in Lab 2.

> ### How this lab works
> - **Concept** cells explain the idea and why it matters.
> - **Challenge** cells contain a scaffold with concise `TODO`s for you to complete.
> - Expand **Hint** for guidance after trying the challenge.
> - Expand **Solution** to review one complete approach.
> - **Checkpoint** questions include expandable **Show answer** sections.
>
> Run cells top to bottom. All data is generated synthetically, so nothing needs to be uploaded.

## Setup &mdash; generate a synthetic `trips` dataset
We build ~1&nbsp;million rows in memory with `spark.range()` and derived columns. `spark.range(N)` creates a DataFrame with a single `id` column distributed across partitions; the `withColumn` calls add fields. Run this cell as-is.

In [ ]:
from pyspark.sql import functions as F

N = 1_000_000
trips = (
    spark.range(N)
    .withColumn("vendorId", (F.col("id") % 3 + 1).cast("int"))
    .withColumn("passengerCount", (F.rand(1) * 5 + 1).cast("int"))
    .withColumn("tripDistance", F.round(F.rand(2) * 20, 2))
    .withColumn("fareAmount", F.round(F.rand(3) * 100 + 3, 2))
    .withColumn("pickupDate", F.expr("date_add(to_date('2024-01-01'), cast(id % 180 as int))"))
    .drop("id")
)
print("Rows:", trips.count())

## Lab 1: Your first DataFrame

A **DataFrame** is a distributed, columnar table. You describe *what* you want with transformations (`select`, `filter`, `groupBy`); Spark decides *how* to compute it.

Key idea to feel here: **transformations are lazy**. Building a chain does nothing until you call an **action** (`show`, `count`, `collect`, `write`).

### Challenge 1.1
Explore the schema, then build `summary`: keep trips longer than 5 miles, group by `vendorId`, and return the **trip count** and **average fare** per vendor.

In [ ]:
# Explore first
trips.printSchema()
trips.show(5)

# TODO: Build the requested summary transformation.

# TODO: Display the result ordered by vendor.

<details>
<summary><b>Hint</b></summary>

<p>Filter with <code>F.col</code>, group on <code>vendorId</code>, then aggregate with <code>F.count</code> and <code>F.avg</code>. Round and alias the average before displaying it.</p>

</details>

<details>
<summary><b>Solution</b></summary>

```python
summary = (
    trips.filter(F.col("tripDistance") > 5)
    .groupBy("vendorId")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fareAmount"), 2).alias("avg_fare"),
    )
)
summary.orderBy("vendorId").show()
```

<p><code>groupBy(...).agg(...)</code> is a <b>wide</b> transformation: it forces a shuffle so rows with the same <code>vendorId</code> end up together. Nothing ran until <code>.show()</code>.</p>

</details>

## Lab 2: Lazy evaluation & the DAG 

Because Spark sees your **whole** chain before running, the **Catalyst** optimizer can prune columns and push filters down. An **action** compiles the plan into a **DAG** of stages; each *wide* transformation (shuffle) starts a new **stage**.

`DataFrame.explain(True)` prints the four plan phases: **parsed &rarr; analyzed &rarr; optimized &rarr; physical**.

### Challenge 2.1
Build a multi-step chain as `plan` without calling an action and inspect its plan. Then trigger execution and use Spark UI -> Jobs -> Stages to locate the shuffle boundary.

In [ ]:
# TODO: Build the lazy transformation plan.
plan = (
    trips
    # TODO
    # TODO
)

# TODO: Print the plan without triggering a job.

# TODO: Trigger the plan, then inspect its stages in the Spark UI.

<details>
<summary><b>Hint</b></summary>

<p>Chain a filter, a grouping, and a count into <code>plan</code>. Inspect it with the extended plan method before calling an action. Then display the result and find the exchange boundary in the Spark UI stages.</p>

</details>

<details>
<summary><b>Solution</b></summary>

```python
plan = (
    trips.filter(F.col("fareAmount") > 50)
    .groupBy("vendorId")
    .count()
)
plan.explain(True)   # still no job in the Spark UI
plan.show()

# Spark UI -> Jobs -> (your job) -> Stages:
#   Stage 1 reads, filters, and partially aggregates
#   Shuffle boundary
#   Stage 2 finishes the aggregation
```

<p>Look for an <code>Exchange</code> node in the physical plan; that is the shuffle and the boundary between stages.</p>

</details>

### From DataFrame to a managed Lakehouse table
When you call `df.write.format("delta").saveAsTable("trips")`, two things happen at once:
1. **Data** is written as Delta files (Parquet + `_delta_log/`) under `Tables/` in the Lakehouse on OneLake.
2. The table is **registered in the metastore**, so it's instantly reachable from Spark, the **SQL analytics endpoint (T-SQL)** and **Power BI (Direct Lake)** — no copies.

That is what "managed table" means: *data on OneLake + a catalog entry*. Keep this in mind as you create the Delta table below.

## Lab 3: Create a Delta table

**Delta Lake** = Parquet data files **+** a transaction log (`_delta_log/`). Every write is an **atomic, versioned commit**. `DESCRIBE HISTORY` shows the log.

### Challenge 3.1
Write `trips` as a **Delta** table named `trips`, overwriting if it exists, then show its history (version, timestamp, operation).

In [ ]:
# TODO: Persist the trips DataFrame as the requested managed table.

# TODO: Display the table history.]

<details>
<summary><b>Hint</b></summary>

<p>Use the DataFrame writer with Delta format, overwrite mode, and a managed-table save. Query <code>DESCRIBE HISTORY</code> and select the requested columns.</p>

</details>

<details>
<summary><b>Solution</b></summary>

```python
trips.write.format("delta").mode("overwrite").saveAsTable("trips")
spark.sql("DESCRIBE HISTORY trips").select("version", "timestamp", "operation").show(truncate=False)
```

<p>Explore <b><code>Tables/dbo/trips</code></b> and its <b><code>_delta_log/</code></b> folder in the Lakehouse Explorer; the JSON commits are the source of truth.</p>

</details>

### Challenge 3.2
Run an `UPDATE` (a new atomic commit) that raises `fareAmount` by 10% for `vendorId = 1`, then re-check the history; a new version should appear.

In [ ]:
# TODO: Apply the requested update to the Delta table.

# TODO: Display the latest table history.

<details>
<summary><b>Hint</b></summary>

<p>Use Spark SQL to update the matching vendor rows and round the adjusted fare. Then query <code>DESCRIBE HISTORY</code> to verify a new operation was committed.</p>

</details>

<details>
<summary><b>Solution</b></summary>

```python
spark.sql("UPDATE trips SET fareAmount = ROUND(fareAmount * 1.1, 2) WHERE vendorId = 1")
spark.sql("DESCRIBE HISTORY trips").select("version", "operation").show()
```

</details>

## Lab 4: Time travel, compaction & OPTIMIZE
Because commits are versioned, you can **time travel**. And because streaming/small writes create **many small files**, `OPTIMIZE` compacts them; `ZORDER` co-locates values for file skipping; `VACUUM` removes stale files.

### Challenge 4.1
Query **version 0** of `trips` (before the Lab 3 update) and compare average fare per vendor with the current version.

In [ ]:
# TODO: Compare the requested historical and current aggregates.

<details>
<summary><b>Hint</b></summary>

<p>Run one SQL aggregation against <code>VERSION AS OF 0</code> and the same aggregation against the current table. Group and order by vendor so the results are easy to compare.</p>

</details>

<details>
<summary><b>Solution</b></summary>

```python
spark.sql('''
  SELECT vendorId, ROUND(AVG(fareAmount),2) AS avg_fare
  FROM trips VERSION AS OF 0
  GROUP BY vendorId ORDER BY vendorId
''').show()

spark.sql('''
  SELECT vendorId, ROUND(AVG(fareAmount),2) AS avg_fare
  FROM trips
  GROUP BY vendorId ORDER BY vendorId
''').show()
```

<p>vendorId 1 should be about 10% higher in the current version. <code>TIMESTAMP AS OF '2024-01-01T00:00:00'</code> works too.</p>

</details>

### Challenge 4.2
Create many small files by appending a small slice 100 times to `trips_small`, then **compact** with `OPTIMIZE` (and `ZORDER BY (vendorId)`).

In [ ]:
%run moduele_1_helpers

In [ ]:
small = trips.limit(200000)
for i in range(100):
    small.write.format("delta").mode("append").saveAsTable("trips_small")

In [ ]:
before = table_file_stats("trips_small")
print_stats(before, "BEFORE OPTIMIZE")

In [ ]:
# TODO: Optimize and ZORDER

In [ ]:
after = table_file_stats("trips_small")
print_stats(after, "AFTER OPTIMIZE")

<details>
<summary><b>Hint</b></summary>

You can use <code>OPTIMIZE</code> and <code>ZORDER BY</code>

</details>

<details>
<summary><b>Solution</b></summary>

```python
spark.sql("OPTIMIZE dbo.trips_small ZORDER BY (vendorId)")

```

</details>

## Checkpoint

Answer from memory, then reveal.

**1. What triggers a Spark job, and what creates a new stage?**

<details>
<summary><b>Show answer</b></summary>

<p>An <b>action</b> (<code>count</code>, <code>show</code>, <code>collect</code>, <code>write</code>) triggers a job. A <b>wide transformation</b> (a shuffle, e.g. <code>groupBy</code>, <code>join</code>, <code>distinct</code>) creates a new <b>stage</b> boundary.</p>

</details>

**2. What is the difference between a narrow and a wide transformation?**

<details>
<summary><b>Show answer</b></summary>

<p><b>Narrow</b> (<code>filter</code>, <code>map</code>, <code>withColumn</code>): each output partition depends on one input partition, with no data movement. <b>Wide</b> (<code>groupBy</code>, <code>join</code>): output partitions depend on many input partitions, requiring a <b>shuffle</b> across the network.</p>

</details>

**3. Where does Delta store the truth about which files belong to a table?**

<details>
<summary><b>Show answer</b></summary>

<p>In the <b><code>_delta_log/</code></b> folder: ordered JSON commits plus periodic checkpoints. The physical file listing is <i>not</i> authoritative; the log is.</p>

</details>

**4. What are `OPTIMIZE` and `ZORDER BY`?**

<details>
<summary><b>Show answer</b></summary>

<p><code>OPTIMIZE</code> compacts <b>many small files</b> into fewer large ones (and <code>ZORDER</code> clusters data for skipping).
</details>

### Cleanup (optional)

In [ ]:
# spark.sql("DROP TABLE IF EXISTS trips")
# spark.sql("DROP TABLE IF EXISTS trips_small")